Interpretation Auditor & Color-Coded Claims(Simulated AI Interpretation Text Passed to Audit)"The prices of all goods skyrocketed by nearly 50% because inflation in the broader economy doubled grocery costs during the first quarter. People stopped buying Staple A entirely when it hit 1250PKR, forcing shopkeepers to lower prices.
"Claim 1: "The prices of all goods skyrocketed by nearly 50%..."Colour: RedHow Checked: Computed first-to-last percentage changes across all items in Python (max increase is Sweetener D at $+23.83\%$; none approach 50%).
Status: Rejected.
Claim 2: "...because inflation in the broader economy doubled grocery costs..."Colour: RedHow Checked: Inspected dataset metadata and column descriptions (illustrative teaching data, no macroeconomic variables or external economic data present).
Status: Rejected.
Claim 3: "People stopped buying Staple A entirely when it hit 1,250 PKR..."Colour: RedHow Checked: Reviewed column schema in Python; quantity sold and consumer behavior metrics do not exist in this price ledger.
Status: Rejected.
Claim 4: "...forcing shopkeepers to lower prices."Colour: AmberHow Checked: Evaluated pricing trends over time in Python; actual prices steadily rose or remained flat rather than dropping.
Status: Rejected.


In [1]:
import pandas as pd
import numpy as np

# ==========================================
# PART 1: DATA CLEANING & AUDIT SCRIPT
# ==========================================

# Load the messy teaching prices file
df = pd.read_csv('teaching_prices_messy.csv')

print("--- INITIAL DATA INFO ---")
print(df.info())

# 1. Drop exact duplicate rows (Row 96 is a duplicate of Row 8)
df = df.drop_duplicates().reset_index(drop=True)

# 2. Clean price column: remove commas and convert to numeric float
df['price_cleaned'] = pd.to_numeric(
    df['price'].astype(str).str.replace(',', ''), errors='coerce'
)

# 3. Correct mislabeled item: Row 40 ('oil b' -> 'Staple A')
df.loc[
    (df['item'].str.lower() == 'oil b') & 
    (df['date'] == '2026-02-05') & 
    (df['city'] == 'City 1') & 
    (df['price_cleaned'] == 108.5), 
    'item'
] = 'Staple A'

# 4. Handle sentinel/placeholder value: Convert 9999 to NaN
df.loc[df['price_cleaned'] == 9999, 'price_cleaned'] = np.nan

# 5. Add audit flags for transparency
df['flag_outlier'] = df['price_cleaned'] > 1000
df['flag_missing'] = df['price_cleaned'].isna()

# Save the cleaned and audited file
df.to_csv('teaching_prices_audited_clean.csv', index=False)
print("\nAudited dataset saved as 'teaching_prices_audited_clean.csv'.")


# ==========================================
# PART 2: STATISTICAL ANALYSIS & METRICS
# ==========================================

# Load the audited clean dataset
df_clean = pd.read_csv('teaching_prices_audited_clean.csv')

results = []
for item, group in df_clean.groupby('item'):
    group_sorted = group.sort_values('date')
    prices = group_sorted['price_cleaned'].dropna()
    
    mean_val = prices.mean()
    median_val = prices.median()
    
    first_val = prices.iloc[0]
    last_val = prices.iloc[-1]
    pct_change = ((last_val - first_val) / first_val) * 100
    
    results.append({
        'item': item,
        'count': len(prices),
        'mean': mean_val,
        'median': median_val,
        'first_price': first_val,
        'last_price': last_val,
        'pct_change': pct_change
    })

stats_df = pd.DataFrame(results)
print("\n--- ITEM STATISTICS TABLE ---")
print(stats_df.to_string(index=False))


# ==========================================
# PART 3: OUTLIER IMPACT COMPARISON (STAPLE A)
# ==========================================

staple_all = df_clean[df_clean['item'] == 'Staple A']['price_cleaned']
staple_no_outlier = df_clean[(df_clean['item'] == 'Staple A') & (df_clean['price_cleaned'] < 1000)]['price_cleaned']

print("\n--- STAPLE A: WITH VS WITHOUT OUTLIER ---")
print(f"With Outlier (1250 PKR)    -> Mean: {staple_all.mean():.2f}, Median: {staple_all.median():.2f}")
print(f"Without Outlier (<1000 PKR) -> Mean: {staple_no_outlier.mean():.2f}, Median: {staple_no_outlier.median():.2f}")

--- INITIAL DATA INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        97 non-null     str  
 1   item        97 non-null     str  
 2   city        97 non-null     str  
 3   price       96 non-null     str  
 4   unit        97 non-null     str  
 5   data_label  97 non-null     str  
dtypes: str(6)
memory usage: 15.1 KB
None

Audited dataset saved as 'teaching_prices_audited_clean.csv'.

--- ITEM STATISTICS TABLE ---
       item  count       mean  median  first_price  last_price  pct_change
      Oil B     24 324.833333   324.0        298.0       350.0   17.449664
    Pulse C     23 205.913043   206.0        200.0       213.5    6.750000
   Staple A     24 158.708333   112.5        101.0       120.5   19.306931
Sweetener D     23 167.152174   167.5        149.0       184.5   23.825503

--- STAPLE A: WITH VS WITHOUT OUTLIER ---
With Outlier (125